<img
  src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D01-finpulse-logo.png"
  alt="FinPulse logo"
/>

# FinPulse Data Pipeline

You work as a junior data engineer at FinPulse, a small financial research team that tracks cryptocurrency markets. The team needs fresh prices when a report is due, but right now someone pulls them by hand.

Your manager wants an automated collector. A script on a remote server should fetch live crypto prices from CoinGecko on a schedule without manual work. You build that script here, then you upload the same file to EC2 and schedule it with cron.

You already know how to call CoinGecko with `requests`, build a `params` dict with `ids` and `vs_currencies`, and check status codes with `raise_for_status()`. This exercise turns that pattern into a production-ready `fetch_prices.py` you can run outside Jupyter.

In [1]:
import time

import requests


def fetch_json_with_backoff(url, params, timeout=30, max_attempts=8):
    """GET with simple 429 backoff for CoinGecko in class and CI."""
    backoff = 2.0
    last = None
    for _ in range(max_attempts):
        last = requests.get(url, params=params, timeout=timeout)
        if last.status_code == 429:
            ra = last.headers.get("Retry-After")
            try:
                wait = float(ra) if ra is not None else backoff
            except (TypeError, ValueError):
                wait = backoff
            time.sleep(min(wait, 60.0))
            backoff = min(backoff * 2, 60.0)
            continue
        last.raise_for_status()
        return last
    last.raise_for_status()
    return last


## Task 1: Fetch live prices for a list of coins

Call the CoinGecko `/simple/price` endpoint for `["bitcoin", "ethereum", "solana"]` and print the raw JSON response.


<Note type="hint">
The CoinGecko simple price endpoint expects two query parameters: `ids` (a comma-separated string of coin slugs, for example `"bitcoin,ethereum"`) and `vs_currencies` (for example `"usd"`). Build a `params` dict with those keys, then call `fetch_json_with_backoff(url, params)` from the previous cell so a crowded classroom does not trip CoinGecko rate limits. Call `.json()` on the response.
</Note>


In [2]:
url = "https://api.coingecko.com/api/v3/simple/price"
params = {
    "ids": "bitcoin,ethereum,solana",           # TODO: comma-separated coin names as a single string
    "vs_currencies": "usd"   # TODO: target currency
}

r = fetch_json_with_backoff(url, params)
print(r.json())


{'bitcoin': {'usd': 83579}, 'ethereum': {'usd': 2647.27}, 'solana': {'usd': 113.8}}


## Task 2: Parse the response into a DataFrame

Transform the JSON response into a `pandas` DataFrame with columns `coin`, `price_usd`, and `fetched_at` (current UTC timestamp as an ISO string).


<Note type="hint">
Build a list of dicts first, one dict per coin, then pass the list to `pd.DataFrame()`. For the timestamp, use `datetime.now(timezone.utc).isoformat()` to get a timezone-aware UTC ISO string.
</Note>


<Note type="tip" title="UTC timestamps">
Always record the fetch time in UTC so the data stays comparable regardless of where the script runs. Converting to a local timezone is a display concern, not a storage one.
</Note>


In [6]:
import pandas as pd
from datetime import datetime, timezone

data = r.json()
# first_dict = []
# for coin, values in data.items():
#     first_dict.append({
#         "coin": coin,
#         "price_usd": values["usd"],
#         "fetched_at": datetime.now(timezone.utc)
#     })
# df = pd.DataFrame(first_dict)
# print(df)

rows = [
    {"coin": coin, "price_usd": values["usd"], "fetched_at": datetime.now(timezone.utc)}   # TODO: fill in the three values per coin
    for coin, values in data.items()                        # TODO: iterate over the response dict
]
df = pd.DataFrame(rows)
print(df)


       coin  price_usd                       fetched_at
0   bitcoin   83579.00 2026-09-24 13:31:21.880605+00:00
1  ethereum    2647.27 2026-09-24 13:31:21.880612+00:00
2    solana     113.80 2026-09-24 13:31:21.880613+00:00


## Task 3: Add error handling and a polite delay

Wrap the request from Task 1 in a `try/except` block. Inside the `try` block, call `fetch_json_with_backoff(url, params)` (it already raises on HTTP errors), add `time.sleep(0.3)` after the response returns, then call `.json()`.


<Note type="hint">
Use `try: ... except requests.exceptions.RequestException as e: ...`. Import `time` is already available from the setup cell, you can still import it again if you want a self-contained block. Put `time.sleep(0.3)` after the successful fetch and before you parse JSON.
</Note>


<Note type="warning">
Skipping `raise_for_status()` means a 429 or 500 response returns silently, bad data can flow downstream without any signal.
</Note>


In [7]:
import time

try:
    r = fetch_json_with_backoff(url, params)
    time.sleep(0.3)  # TODO: add a polite delay (0.3s is enough for CoinGecko)
    data = r.json()
except requests.exceptions.RequestException as e:
    print(f"Error sur la récup de data: {e}")  # TODO: print a clear error message and set data to None
    data = None

if data:
    rows = [
        {"coin": coin, "price_usd": values["usd"], "fetched_at": datetime.now(timezone.utc).isoformat()}
        for coin, values in data.items()
    ]
    df = pd.DataFrame(rows)
    print(df)


       coin  price_usd                        fetched_at
0   bitcoin   84003.00  2026-09-24T13:37:51.122545+00:00
1  ethereum    2666.19  2026-09-24T13:37:51.122566+00:00
2    solana     114.95  2026-09-24T13:37:51.122570+00:00


## Task 4: Save to CSV and package the script

Save the DataFrame to `prices.csv` with `df.to_csv()` (no index). Then use `%%writefile fetch_prices.py` to package everything from Tasks 1 to 3 into a standalone deployable script.


<Note type="hint">
The `%%writefile` cell runs as a plain Python script, it has no access to variables from earlier notebook cells. Copy your working fetch plus parse plus save logic from Tasks 1 to 3, wrap it in a `main()` function, and call `main()` at the bottom so the script runs when invoked directly.
</Note>


<Note type="important">
Keep `fetch_prices.py`. You will upload this file to your EC2 instance when you schedule the job. After the `%%writefile` cell runs, confirm the file exists with `import os; print(os.path.exists("fetch_prices.py"))`.
</Note>


In [8]:
df.to_csv("prices.csv", index=False)  # TODO: filename ("prices.csv") and whether to include the row index


In [9]:
%%writefile fetch_prices.py
import requests
import pandas as pd
from datetime import datetime, timezone
import time


def main():
    url = "https://api.coingecko.com/api/v3/simple/price"
    params = {
        "ids": ___,            # TODO: comma-separated coin names
        "vs_currencies": ___   # TODO: currency
    }
    try:
        r = requests.get(url, params=params)
        ___                    # TODO: raise_for_status
        time.sleep(___)        # TODO: polite delay
        data = r.json()
    except requests.exceptions.RequestException as e:
        ___                    # TODO: print error and return
        return
    rows = [
        {"coin": coin, "price_usd": values["usd"], "fetched_at": datetime.now(timezone.utc).isoformat()}
        for coin, values in data.items()
    ]
    df = pd.DataFrame(rows)
    df.to_csv("prices.csv", index=False)
    print(f"Saved {len(df)} rows to prices.csv")


if __name__ == "__main__":
    main()


Writing fetch_prices.py
